In [1]:
# RGTransformer 训练器
# 说明：每个阶段都可以独立运行，不依赖其他阶段的执行
%reload_ext autoreload
%autoreload 2

# %% 0. 导入必要的库
import sys
import os

sys.path.append('/home/morisi/Workspace/3D-Ocean')

from src.trainer.base import BaseTrainer, BasePrediction
from src.models.SST.RGTransformer import RGTransformer
from src.config.area import Area
from src.config.params import PROJECT_PATH
from src.dataset.ERA5 import ERA5SSTMonthlyDataset
from src.dataset.OISST import OISSTMonthlyDataset

print("✅ 库导入完成")
print(f"项目根目录: {PROJECT_PATH}")


/home/morisi/Workspace/3D-Ocean/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/morisi/Workspace/3D-Ocean/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happen

✅ 库导入完成
项目根目录: /home/morisi/Workspace/3D-Ocean


In [2]:
# %% 全局配置（所有阶段共享）
# 这个cell定义了所有阶段共享的配置参数，每次运行任何阶段前都需要先运行这个cell

area = Area('Global', lon=[-180, 180], lat=[-80, 80], description='全球区域')

# 基础配置参数
resolution = 1
seq_len = 2
offset = 0
use_checkpoint = True

# 计算空间尺寸
width = int(area.width / resolution)
height = int(area.height / resolution)

# 数据集参数
dataset_params = {
    "seq_len": seq_len,
    "offset": offset,
    "resolution": resolution,
}

# 训练器参数（基础配置）
trainer_epochs = 100 # 先定义训练轮数

trainer_params = {
    "epochs": trainer_epochs,
    "batch_size": 32,
    "num_workers": 12,
    "use_wandb": True,
    "use_checkpoint": use_checkpoint,
    "save_top_k": 1,
    "monitor": "val_loss",
    "mode": "min",
}

# 模型参数（基础配置）
rg_transformer_m_params = {
    "width": width,
    "height": height,
    "resolution": resolution,
    "lat_range": area.lat,
    "lon_range": area.lon,
    "seq_len": seq_len,
    "d_model": 1024,
    "num_heads": 8,
    "dim_feedforward": 512,
    "dropout": 0.1,
    "recursion_depth": 2,
    "learning_rate": 1e-3,  # 初始学习率
}

# Checkpoint路径（使用项目根目录的绝对路径）
CHECKPOINT_DIR = f'{PROJECT_PATH}/out/checkpoints'
CHECKPOINT_FILE = 'RGTransformer.ckpt'

print("=" * 70)
print("📋 全局配置")
print("=" * 70)
print(f"区域: {area.title} ({area.description})")
print(f"分辨率: {resolution}°")
print(f"序列长度: {seq_len}")
print(f"空间尺寸: {width} x {height}")
print(f"Checkpoint 文件: {CHECKPOINT_FILE}")
print("=" * 70)

📋 全局配置
区域: Global (全球区域)
分辨率: 1°
序列长度: 2
空间尺寸: 160 x 360
Checkpoint 文件: RGTransformer.ckpt


In [ ]:
# %% 1. 预训练阶段
# 说明：可以独立运行，从头开始训练模型
# 如果已有checkpoint，可以选择继续训练或重新训练

print("=" * 70)
print("🚀 预训练阶段")
print("=" * 70)

print(f"\n模型: RGTransformer")
print(f"训练轮数: {trainer_params['epochs']}")
print(f"学习率: {rg_transformer_m_params['learning_rate']}")
print(f"Checkpoint: {'启用' if use_checkpoint else '禁用'}")
print("=" * 70 + "\n")

# 创建训练器
pretrain_trainer = BaseTrainer(
    area=area,
    model_class=RGTransformer,
    dataset_class=ERA5SSTMonthlyDataset,# 如果要从已有checkpoint继续，设置此参数
    use_checkpoint=use_checkpoint,
    dataset_params=dataset_params,
    trainer_params=trainer_params,
    model_params=rg_transformer_m_params,
)

# 开始训练
pretrain_model = pretrain_trainer.train()

print("\n" + "=" * 70)
print("✅ 预训练完成！")
print("=" * 70)
print(f"最优模型保存在: {CHECKPOINT_DIR}/")
print("=" * 70)

In [ ]:
# %% 2. 验证阶段
# 说明：可以独立运行，评估已训练模型的性能
# 需要先有训练好的checkpoint

print("=" * 70)
print("📊 验证阶段")
print("=" * 70)

# 创建评估训练器（不训练，只加载模型进行评估）
eval_trainer = BasePrediction(
    wandb_run_id="2025-11-09-22-18",
    area=area,
    model_class=RGTransformer,
    dataset_class=ERA5SSTMonthlyDataset,
    dataset_params=dataset_params,
    model_params=rg_transformer_m_params,
)

# 单个时间点详细预测（可选，绘制图表）
print("\n" + "-" * 70)
print("单个时间点详细预测:")
print("-" * 70)


single_result = eval_trainer.predict(offset=520, plot=True)

print("=" * 70)


📊 验证阶段

----------------------------------------------------------------------
单个时间点详细预测:
----------------------------------------------------------------------
📦 从 wandb 加载模型...
  • Run ID: 2025-11-09-22-18
  • Version: latest
  • Project: 3-D Ocean
  • Entity: yiliavei-zhejiang-university
  • 查找 Artifact: yiliavei-zhejiang-university/3-D Ocean/RGTransformer_2025-11-09-22-18:latest


wandb: Downloading large artifact 'RGTransformer_2025-11-09-22-18:latest', 1204.57MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:11.0 (109.6MB/s)


  • 重命名文件: RGTransformer-v4.ckpt -> RGTransformer.ckpt
  • 已初始化延迟投影层


/home/morisi/Workspace/3D-Ocean/.venv/lib/python3.13/site-packages/lightning/pytorch/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['attention.input_proj.weight', 'attention.input_proj.bias', 'attention.output_proj.weight', 'attention.output_proj.bias']


起始时间：1983-05-01
成功加载ERA5数据: (1030, 181, 361)
时间步数: 1030
经度范围: [-180.00, 180.00]
纬度范围: [-90.00, 90.00]
分辨率: 1°
--------------------------------
Model: RGTransformer Prediction RMSE: 1.0286608934402466
--------------------------------
 📊 Model: RGTransformer Prediction Position Encoding:
x_normed: (1, 160, 360)
position_encoding: (160, 360)
attention_out: (1, 160, 360)
ffn_out: (1, 160, 360)
sst_after: (1, 160, 360)
temporal_weights: (1,)
--------------------------------
 📊 Model: RGTransformer Prediction Other Parameters:
spatial_enc_scale: 0.2622416913509369
--------------------------------
NINO3.4 指数: 1.786°C
NINO3 指数: 2.893°C


In [ ]:
# %% 3. 微调阶段
# 说明：可以独立运行，从已有checkpoint继续训练
# 需要先有checkpoint（可以通过预训练或之前的微调获得）

print("=" * 70)
print("✨ 微调阶段")
print("=" * 70)

# 微调参数配置（可以修改）
finetune_learning_rate = 5e-5  # 微调学习率（通常比预训练小）
finetune_epochs = 500           # 微调轮数

# 微调模型参数
finetune_m_params = {
    **rg_transformer_m_params,  # 继承基础配置
    "learning_rate": finetune_learning_rate,  # 微调学习率
}

# 微调训练参数（可以调整 epochs、batch_size 等）
finetune_trainer_params = {
    **trainer_params,  # 继承基础配置
    "epochs": finetune_epochs,  # 微调轮数
    # "batch_size": 64,  # 可以增大 batch size（如果显存允许）
    # "gradient_clip_val": 1.0,  # 可以添加梯度裁剪
}

print("\n" + "=" * 70)
print("✅ 微调完成！")
print("=" * 70)
print("=" * 70)